# MLflow Projects: End-to-end project template

This notebook is divided into small cells with markdown explanation and line-by-line comments. It uses `datasets/customer_churn_500.csv` from this folder.

## 1. Import required libraries

This cell imports MLflow, pandas, NumPy, matplotlib, and scikit-learn utilities.

In [ ]:
# Import os to create local folders and manage paths.
import os

# Import warnings to hide non-critical warning messages during training.
import warnings

# Ignore warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking and model management.
import mlflow

# Import MLflow sklearn integration for logging sklearn models.
import mlflow.sklearn

# Import pandas for reading and processing CSV data.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for creating charts and artifact images.
import matplotlib.pyplot as plt

# Import train_test_split for splitting data into train and test sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for separate numerical and categorical processing.
from sklearn.compose import ColumnTransformer

# Import encoders and scalers for preprocessing.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to combine preprocessing and model into one object.
from sklearn.pipeline import Pipeline

# Import ML models.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import regression metrics.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Configure MLflow

This cell configures MLflow to store runs locally in the current folder under `mlruns`.

In [ ]:
# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Store tracking data locally in this folder.
mlflow.set_tracking_uri("file:./mlruns")

# Set experiment name for this notebook.
mlflow.set_experiment("02_mlflow_projects_06_end_to_end_project_template")

# Define the dataset path used by this notebook.
DATA_PATH = "datasets/customer_churn_500.csv"

# Print confirmation.
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)

## 3. Load and inspect dataset

The dataset contains 500 realtime-style customer records with categorical and numerical columns.

In [ ]:
# Read the CSV dataset from the local datasets folder.
df = pd.read_csv(DATA_PATH)

# Display the first five records.
display(df.head())

# Display dataset shape.
print("Dataset shape:", df.shape)

# Display column names.
print("Columns:", df.columns.tolist())

## 4. Prepare features and target

For classification, we predict `churn`. For regression, we predict `monthly_charges`.

In [ ]:
# Define classification target column.
target_column = "churn"

# Remove customer ID and target column from features.
X = df.drop(columns=["customer_id", target_column])

# Store the classification target.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature summary.
print("Target column:", target_column)
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

## 5. Build preprocessing pipeline

Categorical columns are one-hot encoded. Numerical columns are scaled.

In [ ]:
# Create preprocessing logic for numerical and categorical columns.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # Convert categorical columns into numeric one-hot encoded columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Print the preprocessing object.
preprocessor

## 6. Split the dataset

Training data is used to train the model. Testing data is used to evaluate the model.

In [ ]:
# Split data into training and testing datasets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print split sizes.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Create MLflow Project folder

This creates a project structure with `train.py`, `MLproject`, `requirements.txt`, and dataset copy.

In [ ]:
# Import pathlib for file and folder creation.
from pathlib import Path

# Import shutil to copy dataset file.
import shutil

# Define project folder.
project_dir = Path("sample_realtime_mlflow_project")

# Create project folder.
project_dir.mkdir(exist_ok=True)

# Create datasets folder inside project.
(project_dir / "datasets").mkdir(exist_ok=True)

# Copy dataset into project folder.
shutil.copy(DATA_PATH, project_dir / "datasets" / "customer_churn_500.csv")

# Print created project folder.
print("Project folder:", project_dir.resolve())

## 8. Create training script

This script can run independently and logs metrics into MLflow.

In [ ]:
# Write train.py file.
(project_dir / "train.py").write_text("""
import os
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Remove MLflow project-level experiment ID if it was automatically passed.
# This avoids "Could not find experiment with ID 0" error in local file tracking.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove project-level run ID if any previous run context is passed.
os.environ.pop("MLFLOW_RUN_ID", None)

os.makedirs("mlruns", exist_ok=True)
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("02_mlflow_projects_06_end_to_end_project_template")

df = pd.read_csv("datasets/customer_churn_500.csv")
X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)),
])

with mlflow.start_run(run_name="project_training_run"):
    mlflow.log_param("dataset", "datasets/customer_churn_500.csv")
    mlflow.log_param("model_family", "RandomForestClassifier")
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    mlflow.log_metric("accuracy", accuracy_score(y_test, predictions))
    mlflow.log_metric("precision", precision_score(y_test, predictions, zero_division=0))
    mlflow.log_metric("recall", recall_score(y_test, predictions, zero_division=0))
    mlflow.log_metric("f1_score", f1_score(y_test, predictions, zero_division=0))
    mlflow.sklearn.log_model(pipeline, "model")
""")

# Print confirmation.
print("Created train.py")

## 9. Create MLproject and requirements files

The `MLproject` file defines how to run the project.

In [ ]:
# Create MLproject file.
(project_dir / "MLproject").write_text("""
name: sample_realtime_mlflow_project

entry_points:
  main:
    command: "python train.py"
""")

# Create project requirements file.
(project_dir / "requirements.txt").write_text("mlflow==2.20.1\nscikit-learn\npandas\nnumpy\nmatplotlib\n")

# Display generated files.
print("Generated files:")
for item in project_dir.rglob("*"):
    print(item)

## 10. Run instruction

Run this command from the current folder terminal:

```bash
mlflow run sample_realtime_mlflow_project --experiment-name Project_Realtime_Workflow
```

In [ ]:
# Print the exact command for learners.
print("Run this command from terminal:")
print("mlflow run sample_realtime_mlflow_project --experiment-name Project_Realtime_Workflow")